# 05 - Spark Data Analysis: 20 Business Questions

This notebook answers the 20 business questions using **Spark DataFrame API** over `ANALYTICS.OBT_TRIPS`.

Notes:
- The code is written in Spark, not as native Snowflake SQL queries.
- The Snowflake connector may still optimize some operations internally, but the analytical logic here is expressed in Spark transformations.
- To keep the analysis aligned with the assignment period, the dataframe is filtered to `trip_year` between 2015 and 2025.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import datetime
import glob
import os


def log_step(message):
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def build_spark(app_name):
    jars_dir = os.environ.get('SPARK_JARS_DIR', '/home/jovyan/jars')
    jar_paths = sorted(glob.glob(os.path.join(jars_dir, '*.jar')))
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
    )
    if jar_paths:
        jars_csv = ','.join(jar_paths)
        classpath = ':'.join(jar_paths)
        log_step(f'Loading JARs: {jar_paths}')
        builder = (
            builder
            .config('spark.jars', jars_csv)
            .config('spark.driver.extraClassPath', classpath)
            .config('spark.executor.extraClassPath', classpath)
        )
    else:
        log_step(f'No JARs found in {jars_dir}')
    return builder.getOrCreate()


def assert_snowflake_connector(spark):
    class_names = ['net.snowflake.spark.snowflake.DefaultSource', 'snowflake.DefaultSource']
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found. Run `bash scripts/download_snowflake_jars.sh` from the repo root, restart the container, and rerun this notebook.')


app = build_spark('05_spark_data_analysis')
assert_snowflake_connector(app)

sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

log_step('Notebook 05 Spark ready')


[2026-04-07 22:45:24Z] Loading JARs: ['/home/jovyan/jars/snowflake-jdbc-3.28.0.jar', '/home/jovyan/jars/spark-snowflake_2.12-3.1.8.jar']
[2026-04-07 22:45:26Z] Snowflake connector detected: net.snowflake.spark.snowflake.DefaultSource
[2026-04-07 22:45:26Z] Notebook 05 Spark ready


In [2]:
def normalize_columns(df):
    return df.toDF(*[c.lower() for c in df.columns])


start_year = int(os.environ.get('START_YEAR', '2015'))
end_year = int(os.environ.get('END_YEAR', '2025'))

analysis_cols = [
    'trip_nk', 'taxi_type', 'service_type', 'source_year', 'source_month',
    'trip_date', 'trip_year', 'trip_month', 'trip_day_of_week',
    'pickup_hour', 'pickup_time_band',
    'pu_borough', 'pu_zone', 'do_borough', 'do_zone',
    'route_key',
    'vendor_id', 'vendor_desc',
    'rate_code_desc', 'payment_type_desc',
    'passenger_count',
    'trip_distance_miles', 'trip_duration_minutes', 'avg_speed_mph',
    'fare_amount', 'fare_per_mile',
    'tip_amount', 'tip_pct',
    'tolls_amount', 'congestion_surcharge', 'total_amount'
]

obt = (
    normalize_columns(
        app.read.format('snowflake')
        .options(**sf_option)
        .option('dbtable', 'ANALYTICS.OBT_TRIPS')
        .load()
    )
    .select(*analysis_cols)
    .filter(F.col('taxi_type').isin('yellow', 'green'))
    .filter(F.col('trip_nk').isNotNull())
    .filter(F.col('trip_date').isNotNull())
    .filter(F.col('trip_year').between(start_year, end_year))
    .filter(F.col('source_year').between(start_year, end_year))
)

total_rows = obt.count()
log_step(f'Analysis dataframe rows={total_rows}')
obt.printSchema()


[2026-04-07 22:45:50Z] Analysis dataframe rows=445964
root
 |-- trip_nk: string (nullable = true)
 |-- taxi_type: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- source_year: decimal(4,0) (nullable = true)
 |-- source_month: decimal(2,0) (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- trip_year: decimal(38,0) (nullable = true)
 |-- trip_month: decimal(38,0) (nullable = true)
 |-- trip_day_of_week: string (nullable = true)
 |-- pickup_hour: decimal(38,0) (nullable = true)
 |-- pickup_time_band: string (nullable = false)
 |-- pu_borough: string (nullable = true)
 |-- pu_zone: string (nullable = true)
 |-- do_borough: string (nullable = true)
 |-- do_zone: string (nullable = true)
 |-- route_key: string (nullable = false)
 |-- vendor_id: decimal(38,0) (nullable = true)
 |-- vendor_desc: string (nullable = true)
 |-- rate_code_desc: string (nullable = true)
 |-- payment_type_desc: string (nullable = true)
 |-- passenger_count: double (nullable = true

## Pregunta (a) — Top 10 zonas de pickup por volumen mensual


In [3]:
w_pu = Window.partitionBy('trip_year', 'trip_month').orderBy(F.desc('trip_count'))

df_a = (
    obt
    .groupBy('trip_year', 'trip_month', 'pu_zone')
    .agg(F.count('*').alias('trip_count'))
    .withColumn('rank', F.row_number().over(w_pu))
    .filter(F.col('rank') <= 10)
    .orderBy('trip_year', 'trip_month', 'rank')
)

df_a.show(50, truncate=False)


+---------+----------+------------------------+----------+----+
|trip_year|trip_month|pu_zone                 |trip_count|rank|
+---------+----------+------------------------+----------+----+
|2020     |1         |East Harlem North       |34114     |1   |
|2020     |1         |East Harlem South       |28668     |2   |
|2020     |1         |Central Harlem          |23627     |3   |
|2020     |1         |Elmhurst                |17739     |4   |
|2020     |1         |Astoria                 |15982     |5   |
|2020     |1         |Morningside Heights     |15819     |6   |
|2020     |1         |Central Harlem North    |15090     |7   |
|2020     |1         |Forest Hills            |13549     |8   |
|2020     |1         |Fort Greene             |12596     |9   |
|2020     |1         |Washington Heights South|11085     |10  |
+---------+----------+------------------------+----------+----+



## Pregunta (b) — Top 10 zonas de dropoff por volumen mensual


In [4]:
w_do = Window.partitionBy('trip_year', 'trip_month').orderBy(F.desc('trip_count'))

df_b = (
    obt
    .groupBy('trip_year', 'trip_month', 'do_zone')
    .agg(F.count('*').alias('trip_count'))
    .withColumn('rank', F.row_number().over(w_do))
    .filter(F.col('rank') <= 10)
    .orderBy('trip_year', 'trip_month', 'rank')
)

df_b.show(50, truncate=False)


+---------+----------+---------------------+----------+----+
|trip_year|trip_month|do_zone              |trip_count|rank|
+---------+----------+---------------------+----------+----+
|2020     |1         |East Harlem North    |16335     |1   |
|2020     |1         |Central Harlem North |14952     |2   |
|2020     |1         |Central Harlem       |13240     |3   |
|2020     |1         |East Harlem South    |12433     |4   |
|2020     |1         |Jackson Heights      |10096     |5   |
|2020     |1         |Astoria              |9962      |6   |
|2020     |1         |Morningside Heights  |8628      |7   |
|2020     |1         |Forest Hills         |7404      |8   |
|2020     |1         |Upper East Side North|7245      |9   |
|2020     |1         |Elmhurst             |6820      |10  |
+---------+----------+---------------------+----------+----+



## Pregunta (c) — Evolucion mensual de total_amount y tip_pct por borough


In [5]:
df_c = (
    obt
    .groupBy('trip_year', 'trip_month', 'pu_borough')
    .agg(
        F.round(F.sum('total_amount'), 2).alias('total_revenue'),
        F.round(F.avg('tip_pct'), 2).alias('avg_tip_pct')
    )
    .orderBy('pu_borough', 'trip_year', 'trip_month')
)

df_c.show(100, truncate=False)


+---------+----------+-------------+-------------+-----------+
|trip_year|trip_month|pu_borough   |total_revenue|avg_tip_pct|
+---------+----------+-------------+-------------+-----------+
|2020     |1         |Bronx        |943962.22    |1.29       |
|2020     |1         |Brooklyn     |2712130.16   |8.39       |
|2020     |1         |EWR          |466.61       |20.24      |
|2020     |1         |Manhattan    |2441121.79   |10.67      |
|2020     |1         |N/A          |13741.23     |25.64      |
|2020     |1         |Queens       |2285848.81   |7.36       |
|2020     |1         |Staten Island|11340.2      |0.44       |
|2020     |1         |Unknown      |5716.12      |8.97       |
+---------+----------+-------------+-------------+-----------+



## Pregunta (d) — Ticket promedio (avg total_amount) por service_type y mes


In [6]:
df_d = (
    obt
    .groupBy('taxi_type', 'trip_year', 'trip_month')
    .agg(F.round(F.avg('total_amount'), 2).alias('avg_total_amount'))
    .orderBy('taxi_type', 'trip_year', 'trip_month')
)

df_d.show(100, truncate=False)


+---------+---------+----------+----------------+
|taxi_type|trip_year|trip_month|avg_total_amount|
+---------+---------+----------+----------------+
|green    |2020     |1         |18.87           |
+---------+---------+----------+----------------+



## Pregunta (e) — Viajes por hora del dia y dia de semana (picos)


In [7]:
df_e = (
    obt
    .groupBy('pickup_hour', 'trip_day_of_week')
    .agg(F.count('*').alias('trip_count'))
    .orderBy('pickup_hour', 'trip_day_of_week')
)

df_e.show(200, truncate=False)


+-----------+----------------+----------+
|pickup_hour|trip_day_of_week|trip_count|
+-----------+----------------+----------+
|0          |Fri             |1315      |
|0          |Mon             |771       |
|0          |Sat             |1819      |
|0          |Sun             |1892      |
|0          |Thu             |1124      |
|0          |Tue             |718       |
|0          |Wed             |1731      |
|1          |Fri             |787       |
|1          |Mon             |482       |
|1          |Sat             |1435      |
|1          |Sun             |1506      |
|1          |Thu             |579       |
|1          |Tue             |432       |
|1          |Wed             |1393      |
|2          |Fri             |501       |
|2          |Mon             |312       |
|2          |Sat             |1086      |
|2          |Sun             |1167      |
|2          |Thu             |333       |
|2          |Tue             |226       |
|2          |Wed             |1049

## Pregunta (f) — p50 / p90 de trip_duration_min por borough de pickup


In [8]:
df_f = (
    obt
    .groupBy('pu_borough')
    .agg(
        F.round(F.percentile_approx('trip_duration_minutes', 0.5), 2).alias('p50_duration_min'),
        F.round(F.percentile_approx('trip_duration_minutes', 0.9), 2).alias('p90_duration_min'),
        F.count('*').alias('trip_count')
    )
    .orderBy(F.desc('trip_count'))
)

df_f.show(truncate=False)


+-------------+----------------+----------------+----------+
|pu_borough   |p50_duration_min|p90_duration_min|trip_count|
+-------------+----------------+----------------+----------+
|Manhattan    |10.05           |26.57           |154045    |
|Brooklyn     |15.12           |null            |127839    |
|Queens       |11.13           |null            |126866    |
|Bronx        |null            |null            |36322     |
|Unknown      |0.12            |17.67           |381       |
|N/A          |null            |null            |322       |
|Staten Island|45.73           |66.63           |185       |
|EWR          |0.05            |0.22            |4         |
+-------------+----------------+----------------+----------+



## Pregunta (g) — avg_speed_mph por franja horaria (6-9, 17-20) y borough


In [9]:
df_g = (
    obt
    .withColumn(
        'rush_band',
        F.when((F.col('pickup_hour') >= 6) & (F.col('pickup_hour') <= 9), F.lit('Morning Rush (6-9)'))
        .when((F.col('pickup_hour') >= 17) & (F.col('pickup_hour') <= 20), F.lit('Evening Rush (17-20)'))
    )
    .filter(F.col('rush_band').isNotNull())
    .filter(F.col('avg_speed_mph').isNotNull())
    .filter(F.col('avg_speed_mph') > 0)
    .filter(F.col('avg_speed_mph') < 100)
    .groupBy('pu_borough', 'rush_band')
    .agg(
        F.round(F.avg('avg_speed_mph'), 2).alias('avg_speed_mph'),
        F.count('*').alias('trip_count')
    )
    .orderBy('pu_borough', 'rush_band')
)

df_g.show(truncate=False)


+-------------+--------------------+-------------+----------+
|pu_borough   |rush_band           |avg_speed_mph|trip_count|
+-------------+--------------------+-------------+----------+
|Bronx        |Evening Rush (17-20)|14.84        |5141      |
|Bronx        |Morning Rush (6-9)  |15.65        |8296      |
|Brooklyn     |Evening Rush (17-20)|11.35        |28451     |
|Brooklyn     |Morning Rush (6-9)  |12.87        |18212     |
|Manhattan    |Evening Rush (17-20)|11.97        |37773     |
|Manhattan    |Morning Rush (6-9)  |11.95        |27272     |
|N/A          |Evening Rush (17-20)|21.48        |65        |
|N/A          |Morning Rush (6-9)  |20.43        |47        |
|Queens       |Evening Rush (17-20)|11.98        |31884     |
|Queens       |Morning Rush (6-9)  |14.58        |15391     |
|Staten Island|Evening Rush (17-20)|22.27        |29        |
|Staten Island|Morning Rush (6-9)  |19.4         |13        |
|Unknown      |Evening Rush (17-20)|17.76        |28        |
|Unknown

## Pregunta (h) — Participacion por payment_type_desc y su relacion con tip_pct


In [10]:
df_h = (
    obt
    .groupBy('payment_type_desc')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('tip_pct'), 2).alias('avg_tip_pct'),
        F.round(F.avg('tip_amount'), 2).alias('avg_tip_amount')
    )
    .withColumn('pct_share', F.round(F.col('trip_count') / F.lit(total_rows) * 100, 2))
    .orderBy(F.desc('trip_count'))
)

df_h.show(truncate=False)


+-----------------+----------+-----------+--------------+---------+
|payment_type_desc|trip_count|avg_tip_pct|avg_tip_amount|pct_share|
+-----------------+----------+-----------+--------------+---------+
|Credit card      |184280    |20.04      |2.18          |41.32    |
|Cash             |144776    |0.0        |0.0           |32.46    |
|null             |114626    |0.02       |0.0           |25.7     |
|No charge        |1685      |0.03       |-0.01         |0.38     |
|Dispute          |581       |0.0        |0.0           |0.13     |
|Unknown          |16        |0.0        |0.0           |0.0      |
+-----------------+----------+-----------+--------------+---------+



## Pregunta (i) — ¿Que rate_code_desc concentran mayor trip_distance y total_amount?


In [11]:
df_i = (
    obt
    .groupBy('rate_code_desc')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.sum('trip_distance_miles'), 2).alias('total_distance_miles'),
        F.round(F.sum('total_amount'), 2).alias('total_revenue'),
        F.round(F.avg('trip_distance_miles'), 2).alias('avg_distance'),
        F.round(F.avg('total_amount'), 2).alias('avg_ticket')
    )
    .orderBy(F.desc('total_revenue'))
)

df_i.show(truncate=False)


+---------------------+----------+--------------------+-------------+------------+----------+
|rate_code_desc       |trip_count|total_distance_miles|total_revenue|avg_distance|avg_ticket|
+---------------------+----------+--------------------+-------------+------------+----------+
|Standard rate        |322463    |820430.47           |4728219.94   |2.54        |14.66     |
|null                 |114627    |758371.92           |3451932.43   |6.62        |30.11     |
|Negotiated fare      |7785      |25210.19            |165430.97    |3.24        |21.25     |
|JFK                  |662       |8131.4              |39527.77     |12.28       |59.71     |
|Nassau or Westchester|252       |3691.58             |17182.84     |14.65       |68.19     |
|Newark               |172       |2397.03             |12023.29     |13.94       |69.9      |
|Group ride           |3         |5.2                 |9.9          |1.73        |3.3       |
+---------------------+----------+--------------------+-----

## Pregunta (j) — Mix yellow vs green por mes y borough


In [12]:
df_j_base = (
    obt
    .groupBy('trip_year', 'trip_month', 'pu_borough', 'taxi_type')
    .agg(F.count('*').alias('trip_count'))
)

w_j = Window.partitionBy('trip_year', 'trip_month', 'pu_borough')

df_j = (
    df_j_base
    .withColumn('total_borough', F.sum('trip_count').over(w_j))
    .withColumn('pct_share', F.round(F.col('trip_count') / F.col('total_borough') * 100, 2))
    .orderBy('pu_borough', 'trip_year', 'trip_month', 'taxi_type')
)

df_j.show(100, truncate=False)


+---------+----------+-------------+---------+----------+-------------+---------+
|trip_year|trip_month|pu_borough   |taxi_type|trip_count|total_borough|pct_share|
+---------+----------+-------------+---------+----------+-------------+---------+
|2020     |1         |Bronx        |green    |36322     |36322        |100.0    |
|2020     |1         |Brooklyn     |green    |127839    |127839       |100.0    |
|2020     |1         |EWR          |green    |4         |4            |100.0    |
|2020     |1         |Manhattan    |green    |154045    |154045       |100.0    |
|2020     |1         |N/A          |green    |322       |322          |100.0    |
|2020     |1         |Queens       |green    |126866    |126866       |100.0    |
|2020     |1         |Staten Island|green    |185       |185          |100.0    |
|2020     |1         |Unknown      |green    |381       |381          |100.0    |
+---------+----------+-------------+---------+----------+-------------+---------+



## Pregunta (k) — Top 20 flujos PU -> DO por volumen y su ticket promedio


In [13]:
df_k = (
    obt
    .groupBy('pu_zone', 'do_zone', 'route_key')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('total_amount'), 2).alias('avg_total_amount'),
        F.round(F.avg('trip_distance_miles'), 2).alias('avg_distance'),
        F.round(F.avg('trip_duration_minutes'), 2).alias('avg_duration_min')
    )
    .orderBy(F.desc('trip_count'))
    .limit(20)
)

df_k.show(20, truncate=False)


+--------------------+---------------------+------------------------------------------+----------+----------------+------------+----------------+
|pu_zone             |do_zone              |route_key                                 |trip_count|avg_total_amount|avg_distance|avg_duration_min|
+--------------------+---------------------+------------------------------------------+----------+----------------+------------+----------------+
|East Harlem South   |East Harlem North    |East Harlem South->East Harlem North      |5261      |8.31            |1.2         |8.85            |
|East Harlem North   |East Harlem South    |East Harlem North->East Harlem South      |4843      |9.17            |1.24        |9.32            |
|Astoria             |Astoria              |Astoria->Astoria                          |4608      |7.83            |0.82        |11.50           |
|Forest Hills        |Forest Hills         |Forest Hills->Forest Hills                |4267      |8.61            |0.97     

## Pregunta (l) — Distribucion de passenger_count y efecto en total_amount


In [14]:
df_l = (
    obt
    .filter(F.col('passenger_count').isNotNull())
    .filter(F.col('passenger_count') >= 0)
    .filter(F.col('passenger_count') <= 9)
    .groupBy('passenger_count')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('total_amount'), 2).alias('avg_total_amount'),
        F.round(F.avg('trip_distance_miles'), 2).alias('avg_distance'),
        F.round(F.avg('tip_pct'), 2).alias('avg_tip_pct')
    )
    .orderBy('passenger_count')
)

df_l.show(truncate=False)


+---------------+----------+----------------+------------+-----------+
|passenger_count|trip_count|avg_total_amount|avg_distance|avg_tip_pct|
+---------------+----------+----------------+------------+-----------+
|0.0            |505       |12.85           |1.84        |10.31      |
|1.0            |285381    |14.95           |2.57        |11.2       |
|2.0            |24673     |15.58           |2.87        |11.29      |
|3.0            |4896      |15.13           |2.83        |9.31       |
|4.0            |1498      |15.32           |2.75        |10.65      |
|5.0            |9581      |14.38           |2.57        |12.13      |
|6.0            |4782      |14.51           |2.6         |11.6       |
|7.0            |7         |36.0            |3.28        |10.43      |
|8.0            |13        |24.64           |1.04        |11.36      |
|9.0            |2         |10.23           |1.1         |10.33      |
+---------------+----------+----------------+------------+-----------+



## Pregunta (m) — Impacto de tolls_amount y congestion_surcharge por zona


In [15]:
df_m = (
    obt
    .groupBy('pu_zone')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('tolls_amount'), 2).alias('avg_tolls'),
        F.round(F.sum('tolls_amount'), 2).alias('total_tolls'),
        F.round(F.avg('congestion_surcharge'), 2).alias('avg_congestion_surcharge'),
        F.round(F.sum('congestion_surcharge'), 2).alias('total_congestion_surcharge')
    )
    .filter(F.col('trip_count') > 1000)
    .orderBy(F.desc('avg_tolls'))
)

df_m.show(30, truncate=False)


+---------------------------------+----------+---------+-----------+------------------------+--------------------------+
|pu_zone                          |trip_count|avg_tolls|total_tolls|avg_congestion_surcharge|total_congestion_surcharge|
+---------------------------------+----------+---------+-----------+------------------------+--------------------------+
|Richmond Hill                    |1317      |1.72     |2263.59    |0.03                    |16.5                      |
|Co-Op City                       |1811      |1.45     |2620.92    |0.02                    |2.75                      |
|Soundview/Castle Hill            |1809      |1.34     |2418.5     |0.02                    |8.25                      |
|Williamsbridge/Olinville         |1299      |1.23     |1594.73    |0.0                     |0.0                       |
|East Tremont                     |1008      |1.07     |1082.52    |0.07                    |24.75                     |
|Soundview/Bruckner             

## Pregunta (n) — Proporcion de viajes cortos vs largos por borough y estacionalidad


In [16]:
df_n = (
    obt
    .withColumn(
        'trip_length_cat',
        F.when(F.col('trip_distance_miles') < 3, F.lit('Short (<3 mi)'))
        .otherwise(F.lit('Long (>=3 mi)'))
    )
    .withColumn(
        'season',
        F.when(F.col('trip_month').isin(12, 1, 2), F.lit('Winter'))
        .when(F.col('trip_month').isin(3, 4, 5), F.lit('Spring'))
        .when(F.col('trip_month').isin(6, 7, 8), F.lit('Summer'))
        .otherwise(F.lit('Fall'))
    )
    .groupBy('pu_borough', 'season', 'trip_length_cat')
    .agg(F.count('*').alias('trip_count'))
)

w_n = Window.partitionBy('pu_borough', 'season')

df_n = (
    df_n
    .withColumn('total_in_group', F.sum('trip_count').over(w_n))
    .withColumn('pct', F.round(F.col('trip_count') / F.col('total_in_group') * 100, 2))
    .orderBy('pu_borough', 'season', 'trip_length_cat')
)

df_n.show(50, truncate=False)


+-------------+------+---------------+----------+--------------+-----+
|pu_borough   |season|trip_length_cat|trip_count|total_in_group|pct  |
+-------------+------+---------------+----------+--------------+-----+
|Bronx        |Winter|Long (>=3 mi)  |21178     |36322         |58.31|
|Bronx        |Winter|Short (<3 mi)  |15144     |36322         |41.69|
|Brooklyn     |Winter|Long (>=3 mi)  |55774     |127839        |43.63|
|Brooklyn     |Winter|Short (<3 mi)  |72065     |127839        |56.37|
|EWR          |Winter|Short (<3 mi)  |4         |4             |100.0|
|Manhattan    |Winter|Long (>=3 mi)  |41691     |154045        |27.06|
|Manhattan    |Winter|Short (<3 mi)  |112354    |154045        |72.94|
|N/A          |Winter|Long (>=3 mi)  |217       |322           |67.39|
|N/A          |Winter|Short (<3 mi)  |105       |322           |32.61|
|Queens       |Winter|Long (>=3 mi)  |45846     |126866        |36.14|
|Queens       |Winter|Short (<3 mi)  |81020     |126866        |63.86|
|State

## Pregunta (o) — Diferencias por vendor en avg_speed_mph y trip_duration_min


In [17]:
df_o = (
    obt
    .filter(F.col('avg_speed_mph').isNotNull())
    .filter(F.col('avg_speed_mph') > 0)
    .filter(F.col('avg_speed_mph') < 100)
    .groupBy('vendor_id', 'vendor_desc')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('avg_speed_mph'), 2).alias('avg_speed_mph'),
        F.round(F.avg('trip_duration_minutes'), 2).alias('avg_duration_min'),
        F.round(F.avg('trip_distance_miles'), 2).alias('avg_distance'),
        F.round(F.avg('total_amount'), 2).alias('avg_total_amount')
    )
    .orderBy('vendor_id')
)

df_o.show(truncate=False)


+---------+----------------------------+----------+-------------+----------------+------------+----------------+
|vendor_id|vendor_desc                 |trip_count|avg_speed_mph|avg_duration_min|avg_distance|avg_total_amount|
+---------+----------------------------+----------+-------------+----------------+------------+----------------+
|1        |Creative Mobile Technologies|45280     |11.87        |12.37           |2.54        |13.94           |
|2        |VeriFone Inc.               |384614    |13.05        |21.11           |3.9         |19.4            |
+---------+----------------------------+----------+-------------+----------------+------------+----------------+



## Pregunta (p) — Relacion metodo de pago <-> tip_amount por hora


In [18]:
df_p = (
    obt
    .filter(F.col('payment_type_desc').isNotNull())
    .groupBy('payment_type_desc', 'pickup_hour')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('tip_amount'), 2).alias('avg_tip_amount'),
        F.round(F.avg('tip_pct'), 2).alias('avg_tip_pct')
    )
    .orderBy('payment_type_desc', 'pickup_hour')
)

df_p.show(100, truncate=False)


+-----------------+-----------+----------+--------------+-----------+
|payment_type_desc|pickup_hour|trip_count|avg_tip_amount|avg_tip_pct|
+-----------------+-----------+----------+--------------+-----------+
|Cash             |0          |4218      |0.0           |0.0        |
|Cash             |1          |3164      |0.0           |0.0        |
|Cash             |2          |2282      |0.0           |0.0        |
|Cash             |3          |2010      |0.0           |0.0        |
|Cash             |4          |1819      |0.0           |0.0        |
|Cash             |5          |1186      |0.0           |0.0        |
|Cash             |6          |1582      |0.0           |0.0        |
|Cash             |7          |3470      |0.0           |0.0        |
|Cash             |8          |5636      |0.0           |0.0        |
|Cash             |9          |6573      |0.0           |0.0        |
|Cash             |10         |7218      |0.0           |0.0        |
|Cash             |1

## Pregunta (q) — Zonas con percentil 99 de duracion/distancia fuera de rango (posible congestion/eventos)


In [19]:
df_q = (
    obt
    .filter(F.col('trip_duration_minutes').isNotNull())
    .filter(F.col('trip_distance_miles').isNotNull())
    .groupBy('pu_zone')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.percentile_approx('trip_duration_minutes', 0.99), 2).alias('p99_duration_min'),
        F.round(F.percentile_approx('trip_distance_miles', 0.99), 2).alias('p99_distance_miles'),
        F.round(F.avg('trip_duration_minutes'), 2).alias('avg_duration_min'),
        F.round(F.avg('trip_distance_miles'), 2).alias('avg_distance_miles')
    )
    .filter(F.col('trip_count') > 1000)
    .orderBy(F.desc('p99_duration_min'))
)

df_q.show(30, truncate=False)


+---------------------------------+----------+----------------+------------------+----------------+------------------+
|pu_zone                          |trip_count|p99_duration_min|p99_distance_miles|avg_duration_min|avg_distance_miles|
+---------------------------------+----------+----------------+------------------+----------------+------------------+
|Steinway                         |5524      |195.53          |14.59             |22.43           |2.36              |
|Canarsie                         |2230      |86.77           |25.4              |27.67           |6.28              |
|Flushing                         |3551      |84.67           |19.25             |25.70           |4.57              |
|Flatbush/Ditmas Park             |2328      |79.38           |22.64             |24.35           |4.84              |
|Borough Park                     |1080      |77.82           |24.61             |26.88           |4.89              |
|Sunnyside                        |2918      |76

## Pregunta (r) — Yield por milla (total_amount / trip_distance) por borough y hora


In [20]:
df_r = (
    obt
    .filter(F.col('trip_distance_miles') > 0)
    .filter(F.col('fare_per_mile').isNotNull())
    .filter(F.col('fare_per_mile') < 100)
    .groupBy('pu_borough', 'pickup_hour')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('fare_per_mile'), 2).alias('avg_yield_per_mile'),
        F.round(F.avg('total_amount'), 2).alias('avg_total_amount')
    )
    .orderBy('pu_borough', 'pickup_hour')
)

df_r.show(200, truncate=False)


+-------------+-----------+----------+------------------+----------------+
|pu_borough   |pickup_hour|trip_count|avg_yield_per_mile|avg_total_amount|
+-------------+-----------+----------+------------------+----------------+
|Bronx        |0          |440       |5.85              |21.51           |
|Bronx        |1          |291       |5.78              |18.84           |
|Bronx        |2          |164       |6.21              |20.48           |
|Bronx        |3          |184       |5.1               |26.31           |
|Bronx        |4          |351       |5.87              |26.87           |
|Bronx        |5          |701       |5.16              |32.21           |
|Bronx        |6          |1314      |4.85              |34.27           |
|Bronx        |7          |1956      |5.92              |29.54           |
|Bronx        |8          |2295      |6.47              |26.38           |
|Bronx        |9          |2711      |6.66              |25.24           |
|Bronx        |10        

## Pregunta (s) — Cambios YoY en volumen y ticket promedio por service_type


In [21]:
df_s_base = (
    obt
    .groupBy('taxi_type', 'trip_year')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('total_amount'), 2).alias('avg_ticket')
    )
)

w_s = Window.partitionBy('taxi_type').orderBy('trip_year')

df_s = (
    df_s_base
    .withColumn('prev_count', F.lag('trip_count').over(w_s))
    .withColumn('prev_ticket', F.lag('avg_ticket').over(w_s))
    .withColumn(
        'yoy_volume_pct',
        F.round((F.col('trip_count') - F.col('prev_count')) / F.col('prev_count') * 100, 2)
    )
    .withColumn(
        'yoy_ticket_pct',
        F.round((F.col('avg_ticket') - F.col('prev_ticket')) / F.col('prev_ticket') * 100, 2)
    )
    .orderBy('taxi_type', 'trip_year')
)

df_s.show(30, truncate=False)


+---------+---------+----------+----------+----------+-----------+--------------+--------------+
|taxi_type|trip_year|trip_count|avg_ticket|prev_count|prev_ticket|yoy_volume_pct|yoy_ticket_pct|
+---------+---------+----------+----------+----------+-----------+--------------+--------------+
|green    |2020     |445964    |18.87     |null      |null       |null          |null          |
+---------+---------+----------+----------+----------+-----------+--------------+--------------+



## Pregunta (t) — Dias con alta congestion_surcharge: efecto en total_amount vs dias normales


In [22]:
df_daily_cong = (
    obt
    .filter(F.col('congestion_surcharge').isNotNull())
    .groupBy('trip_date')
    .agg(
        F.count('*').alias('trip_count'),
        F.round(F.avg('congestion_surcharge'), 4).alias('avg_daily_congestion'),
        F.round(F.avg('total_amount'), 2).alias('avg_daily_total')
    )
)

median_cong = df_daily_cong.select(
    F.percentile_approx('avg_daily_congestion', 0.5).alias('median_cong')
).collect()[0]['median_cong']

print(f'Median daily avg congestion surcharge: {median_cong}')

df_t = (
    df_daily_cong
    .withColumn(
        'congestion_class',
        F.when(F.col('avg_daily_congestion') > median_cong, F.lit('High Congestion'))
        .otherwise(F.lit('Normal'))
    )
    .groupBy('congestion_class')
    .agg(
        F.count('*').alias('num_days'),
        F.round(F.avg('trip_count'), 0).alias('avg_daily_trips'),
        F.round(F.avg('avg_daily_total'), 2).alias('avg_total_amount'),
        F.round(F.avg('avg_daily_congestion'), 4).alias('avg_congestion_surcharge')
    )
)

df_t.show(truncate=False)


Median daily avg congestion surcharge: 0.4796
+----------------+--------+---------------+----------------+------------------------+
|congestion_class|num_days|avg_daily_trips|avg_total_amount|avg_congestion_surcharge|
+----------------+--------+---------------+----------------+------------------------+
|High Congestion |15      |10738.0        |15.09           |0.4983                  |
|Normal          |16      |10642.0        |14.87           |0.4495                  |
+----------------+--------+---------------+----------------+------------------------+



---
## Summary

All 20 business questions are expressed with Spark DataFrame API over `ANALYTICS.OBT_TRIPS`.
This notebook is intended as the rubric-aligned Spark version of the business analysis deliverable.
